# LeetCode 853 — Car Fleet bằng Python

Link đề: https://leetcode.com/problems/car-fleet/ · Độ khó: **Medium** · Chủ đề: Array, Sorting (sắp xếp), Stack (ngăn xếp đơn điệu)

Skeleton LeetCode:

```python
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        ...
```

Ghi chú: bài mô tả một **tình huống vật lý** (xe không được vượt nhau, xe nhanh đuổi kịp xe chậm thì chạy chung với tốc độ xe chậm) nhưng lời giải chỉ cần **thời gian về đích của từng xe**: xe phía sau nào về đích **không muộn hơn** một xe phía trước thì sẽ bị chặn lại và nhập đoàn. Notebook đi từ so sánh từng cặp, sắp xếp + đường biên thời gian, so sánh số nguyên không dùng số thực, ngăn xếp đơn điệu, tới **mô phỏng sự kiện chính xác** để làm oracle.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: các xe cùng chạy về mốc `target` trên một đường thẳng, không được vượt nhau; xe đuổi kịp xe trước thì chạy cùng với tốc độ xe chậm; một **đoàn xe** (car fleet) là một xe hoặc một nhóm xe chạy sát nhau; đếm số đoàn về tới đích.
2. Nắm các công cụ Python cần thiết: `zip` và `sorted` với `key` để **ghép đúng cặp** `(vị trí, tốc độ)`, duyệt ngược, ngăn xếp (`list`), so sánh phân số bằng **nhân chéo** (không dùng số thực), và `fractions.Fraction` cho mô phỏng chính xác.
3. Rút ra **thời gian về đích** `t = (target - position) / speed` và quy tắc: xe phía sau có `t` **nhỏ hơn hoặc bằng** thời gian của xe (hoặc đoàn) đứng ngay trước thì **bị nhập vào** đoàn đó.
4. Đi qua 5 cách: so từng cặp, sắp xếp rồi theo dõi thời gian lớn nhất đã gặp, so sánh nhân chéo bằng số nguyên, ngăn xếp đơn điệu, và mô phỏng sự kiện bằng `Fraction`.
5. Tránh các lỗi kinh điển: quên sắp xếp theo vị trí, sắp xếp riêng `position` và `speed` làm mất cặp, dùng `>=` thay vì `>` (đoàn gặp nhau đúng tại đích vẫn là một đoàn), chia nguyên `//`, và duyệt/đếm sai chiều.

## 2. Đề bài & ví dụ

Có `n` chiếc xe ở các vị trí cho trước cách điểm xuất phát (mốc 0) và đều đi về mốc `target`.

Cho hai mảng độ dài `n`: `position[i]` là vị trí xuất phát của xe `i` (đơn vị dặm) và `speed[i]` là tốc độ của xe `i` (dặm/giờ).

Một xe **không thể vượt** xe khác, nhưng có thể **đuổi kịp** rồi chạy sát cạnh với tốc độ của xe chậm hơn.

Một **đoàn xe** là một xe hoặc một nhóm xe chạy sát nhau. Tốc độ của đoàn là **tốc độ nhỏ nhất** của các xe trong đoàn. Nếu một xe đuổi kịp một đoàn **đúng tại mốc `target`** thì nó vẫn được tính là thuộc đoàn đó.

Trả về **số đoàn xe** sẽ về tới đích.

**Ví dụ 1**

- Input: `target = 12`, `position = [10,8,0,5,3]`, `speed = [2,4,1,1,3]`
- Output: `3`
- Giải thích: xe ở 10 (tốc độ 2) và xe ở 8 (tốc độ 4) gặp nhau đúng tại 12: một đoàn. Xe ở 0 (tốc độ 1) không đuổi kịp ai: một đoàn riêng. Xe ở 5 (tốc độ 1) và xe ở 3 (tốc độ 3) gặp nhau ở 6 rồi đi tốc độ 1: một đoàn.

**Ví dụ 2**

- Input: `target = 10`, `position = [3]`, `speed = [3]`
- Output: `1`

**Ví dụ 3**

- Input: `target = 100`, `position = [0,2,4]`, `speed = [4,2,1]`
- Output: `1`
- Giải thích: xe ở 0 (tốc độ 4) và xe ở 2 (tốc độ 2) gặp nhau ở 4; rồi đoàn (tốc độ 2) đuổi kịp xe ở 4 (tốc độ 1) ở 6; cả ba chạy với tốc độ 1.

**Ràng buộc:**

- `n == position.length == speed.length`, `1 <= n <= 10^5`
- `0 < target <= 10^6`
- `0 <= position[i] < target`; mọi `position[i]` **khác nhau**.
- `0 < speed[i] <= 10^6`

In [1]:
from typing import List
from fractions import Fraction
import random, time

examples = [
    (12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3], 3),
    (10, [3], [3], 1),
    (100, [0, 2, 4], [4, 2, 1], 1),
]
for target, position, speed, expected in examples:
    print(f"target={target} position={position} speed={speed} kỳ vọng={expected}")

target=12 position=[10, 8, 0, 5, 3] speed=[2, 4, 1, 1, 3] kỳ vọng=3
target=10 position=[3] speed=[3] kỳ vọng=1
target=100 position=[0, 2, 4] speed=[4, 2, 1] kỳ vọng=1


## 3. Kiến thức nền

### 3.1. Ghép cặp và sắp xếp theo vị trí

Mỗi xe có **cặp** `(vị trí, tốc độ)`. Khi sắp xếp theo vị trí, tốc độ phải **đi theo** xe của nó. `sorted(zip(position, speed))` sắp xếp các cặp theo phần tử đầu (vị trí). **Sai**: `sorted(position)` và `sorted(speed)` riêng rẽ, vì mất liên kết.

In [2]:
position = [10, 8, 0, 5, 3]
speed = [2, 4, 1, 1, 3]
print("ghép đúng:", sorted(zip(position, speed)))
print("sort riêng (sai):", list(zip(sorted(position), sorted(speed))))
cars_front_first = sorted(zip(position, speed), reverse=True)
print("xe phía trước trước:", cars_front_first)

ghép đúng: [(0, 1), (3, 3), (5, 1), (8, 4), (10, 2)]
sort riêng (sai): [(0, 1), (3, 1), (5, 2), (8, 3), (10, 4)]
xe phía trước trước: [(10, 2), (8, 4), (5, 1), (3, 3), (0, 1)]


### 3.2. Thời gian về đích

Xe ở `p` với tốc độ `s` (nếu chạy tự do) về đích `target` sau `t = (target - p) / s` giờ. Xe **gần đích hơn** (`p` lớn hơn) chưa chắc về **sớm hơn** (có thể rất chậm).

In [3]:
target = 12
for p, s in sorted(zip(position, speed), reverse=True):
    print(f"xe ở {p:2} tốc độ {s}: thời gian tự do về đích = ({target} - {p}) / {s} = {(target - p) / s:.4f}")

xe ở 10 tốc độ 2: thời gian tự do về đích = (12 - 10) / 2 = 1.0000
xe ở  8 tốc độ 4: thời gian tự do về đích = (12 - 8) / 4 = 1.0000
xe ở  5 tốc độ 1: thời gian tự do về đích = (12 - 5) / 1 = 7.0000
xe ở  3 tốc độ 3: thời gian tự do về đích = (12 - 3) / 3 = 3.0000
xe ở  0 tốc độ 1: thời gian tự do về đích = (12 - 0) / 1 = 12.0000


### 3.3. Quy tắc nhập đoàn

Xét hai xe `A` (phía trước) và `B` (phía sau, `p_B < p_A`). Nếu `t_B <= t_A` (B về đích không muộn hơn A khi cả hai chạy tự do) thì B **đuổi kịp** A không muộn hơn lúc tới đích, nên **bị A chặn lại** và nhập đoàn (kể cả gặp đúng tại đích). Nếu `t_B > t_A` thì B không bao giờ đuổi kịp A trước đích: B ở đoàn khác.

In [4]:
def merges(target, A, B):
    """A = (p_A, s_A) ở phía trước, B = (p_B, s_B) ở phía sau (p_B < p_A)."""
    tA = Fraction(target - A[0], A[1])
    tB = Fraction(target - B[0], B[1])
    return tB <= tA, tA, tB

print(merges(12, (10, 2), (8, 4)))        # cùng về đích lúc 1 -> nhập đoàn
print(merges(12, (5, 1), (3, 3)))         # tA = 7, tB = 3 -> nhập đoàn
print(merges(12, (8, 4), (0, 1)))         # tA = 1, tB = 12 -> không

(True, Fraction(1, 1), Fraction(1, 1))
(True, Fraction(7, 1), Fraction(3, 1))
(False, Fraction(1, 1), Fraction(12, 1))


### 3.4. So sánh phân số bằng nhân chéo

Để tránh số thực, so sánh `(T - p1) / s1` với `(T - p2) / s2` bằng cách nhân chéo (mọi mẫu dương): `(T - p1) * s2` so với `(T - p2) * s1`. Mọi số là số nguyên (`<= 10^12`).

In [5]:
T = 12
a, b = (10, 2), (8, 4)
lhs, rhs = (T - a[0]) * b[1], (T - b[0]) * a[1]
print("t_a < t_b ?", lhs < rhs, "| t_a == t_b ?", lhs == rhs, "|", (T - a[0]) / a[1], (T - b[0]) / b[1])

t_a < t_b ? False | t_a == t_b ? True | 1.0 1.0


### 3.5. Ngăn xếp (stack) bằng `list`

`append(x)` đẩy vào đỉnh, `pop()` lấy khỏi đỉnh, `st[-1]` xem đỉnh. Ở Cách 4 ta dùng một **ngăn xếp đơn điệu** (mỗi phần tử là thời gian về đích của một đoàn).

In [6]:
st = []
for x in [12, 3, 7, 1, 1]:
    while st and st[-1] <= x:
        st.pop()
    st.append(x)
    print(f"x={x:2}: stack = {st}")

x=12: stack = [12]
x= 3: stack = [12, 3]
x= 7: stack = [12, 7]
x= 1: stack = [12, 7, 1]
x= 1: stack = [12, 7, 1]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `target = 12`. Sắp xếp xe theo vị trí **giảm dần** (từ xe gần đích nhất):

| vị trí | tốc độ | `t = (12 - p) / s` |
|---|---|---|
| 10 | 2 | 1 |
| 8 | 4 | 1 |
| 5 | 1 | 7 |
| 3 | 3 | 3 |
| 0 | 1 | 12 |

Duyệt từ trước ra sau, giữ `pre` = thời gian về đích của **đoàn dẫn đầu gần nhất**:

- xe 10: `t = 1 > 0` -> **đoàn mới**, `pre = 1`.
- xe 8: `t = 1`, không `> pre = 1` -> nhập đoàn xe 10 (gặp nhau đúng tại đích).
- xe 5: `t = 7 > 1` -> **đoàn mới**, `pre = 7`.
- xe 3: `t = 3`, không `> 7` -> nhập đoàn xe 5 (bị xe 5 chặn).
- xe 0: `t = 12 > 7` -> **đoàn mới**, `pre = 12`.

3 đoàn. Quy tắc: xe là **đầu đoàn mới** khi thời gian của nó **lớn hơn mọi thời gian** của các xe phía trước.

In [7]:
target, position, speed, expected = examples[0]
pre = 0
count = 0
for p, s in sorted(zip(position, speed), reverse=True):
    t = (target - p) / s
    if t > pre:
        count += 1
        pre = t
        print(f"xe ở {p:2}: t = {t:5.2f} > pre -> đoàn mới (#{count}), pre = {pre:.2f}")
    else:
        print(f"xe ở {p:2}: t = {t:5.2f} <= pre = {pre:.2f} -> nhập đoàn")
assert count == expected

xe ở 10: t =  1.00 > pre -> đoàn mới (#1), pre = 1.00
xe ở  8: t =  1.00 <= pre = 1.00 -> nhập đoàn
xe ở  5: t =  7.00 > pre -> đoàn mới (#2), pre = 7.00
xe ở  3: t =  3.00 <= pre = 7.00 -> nhập đoàn
xe ở  0: t = 12.00 > pre -> đoàn mới (#3), pre = 12.00


## 5. Cách 1 — Brute Force: so từng cặp

**Ý tưởng.** Tính `t[i]` cho mọi xe. Xe `i` là **đầu một đoàn mới** nếu **không tồn tại** xe `j` ở phía trước (`position[j] > position[i]`) có `t[j] >= t[i]`. Đếm các xe đầu đoàn. Cách này `O(n^2)` và **không cần sắp xếp**.

In [8]:
def fleets_pairs(target: int, position: List[int], speed: List[int]) -> int:
    n = len(position)
    t = [Fraction(target - position[i], speed[i]) for i in range(n)]       # thời gian chính xác
    leaders = 0
    for i in range(n):
        blocked = any(position[j] > position[i] and t[j] >= t[i] for j in range(n))
        if not blocked:
            leaders += 1
    return leaders

for target, position, speed, expected in examples:
    got = fleets_pairs(target, position, speed)
    print(position, speed, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[10, 8, 0, 5, 3] [2, 4, 1, 1, 3] -> 3 OK
[3] [3] -> 1 OK
[0, 2, 4] [4, 2, 1] -> 1 OK


**Độ phức tạp và nút thắt.** `O(n^2)` thời gian: với `n = 10^5` là `10^10`. Nút thắt: ta so mỗi xe với **mọi** xe phía trước, trong khi chỉ cần so với **thời gian lớn nhất** trong số đó (một giá trị duy nhất). Sắp xếp theo vị trí giúp duy trì giá trị lớn nhất đó khi duyệt.

In [9]:
def make_cars(n, target, rng, max_speed=10 ** 6):
    pos = rng.sample(range(target), n)
    spd = [rng.randint(1, max_speed) for _ in range(n)]
    return target, pos, spd

rng = random.Random(1)
for n in (200, 400, 800, 1600):
    target, pos, spd = make_cars(n, 10 ** 6, rng)
    t0 = time.perf_counter()
    fleets_pairs(target, pos, spd)
    print(f"n={n:5}: so từng cặp (Fraction) mất {1000 * (time.perf_counter() - t0):9.1f} ms")

n=  200: so từng cặp (Fraction) mất       1.7 ms
n=  400: so từng cặp (Fraction) mất       3.6 ms
n=  800: so từng cặp (Fraction) mất       5.8 ms
n= 1600: so từng cặp (Fraction) mất      25.0 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp theo vị trí rồi giữ "thời gian lớn nhất đã gặp".** Duyệt xe từ **gần đích nhất** tới xa nhất. Gọi `pre` là thời gian về đích lớn nhất trong các xe đã duyệt (cũng là thời gian của **đoàn dẫn đầu gần nhất** vì mọi xe sau đó bị chặn lại). Xe hiện tại:

- `t > pre`: không bao giờ đuổi kịp bất kỳ xe nào phía trước (kể cả xe có thời gian lớn nhất): **đoàn mới**, `pre = t`.
- `t <= pre`: bị chặn lại: nhập một đoàn có sẵn.

Số đoàn = số lần `pre` tăng ngặt.

Biến thể triển khai:

| Cách | Ý tưởng |
|---|---|
| 2 | Sắp xếp giảm dần theo vị trí, theo dõi `pre` bằng số thực |
| 3 | Như Cách 2 nhưng **so sánh nhân chéo** bằng số nguyên (không số thực) |
| 4 | Sắp xếp tăng dần theo vị trí, ngăn xếp đơn điệu: xe mới (ở phía trước) loại bỏ các đoàn phía sau về đích không muộn hơn |
| 5 | **Mô phỏng sự kiện** (các lần đuổi kịp theo thời gian) bằng `Fraction` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Sắp xếp + theo dõi thời gian lớn nhất (lời giải chuẩn)

**Giải thích từng đoạn.**

- `idx = sorted(range(n), key=lambda i: position[i])` sắp **chỉ số** theo vị trí tăng dần (giữ cặp). Duyệt `idx[::-1]` để đi từ xe gần đích nhất.
- Với mỗi xe: `t = (target - position[i]) / speed[i]`. Nếu `t > pre`: `ans += 1`, `pre = t`.
- Số thực an toàn ở đây: các thời gian là phân số có tử `<= 10^6`, mẫu `<= 10^6`; hai phân số khác nhau cách nhau ít nhất `1/(10^12)` còn khoảng cách tương đối lớn hơn `10^-12`, trội hơn nhiều so với sai số `~10^-16` của số thực kép (mục 10 chứng minh và ta kiểm tra bằng so sánh chính xác).

In [10]:
def fleets_sorted(target: int, position: List[int], speed: List[int]) -> int:
    idx = sorted(range(len(position)), key=lambda i: position[i])
    ans = 0
    pre = 0.0
    for i in idx[::-1]:
        t = (target - position[i]) / speed[i]
        if t > pre:
            ans += 1
            pre = t
    return ans

for target, position, speed, expected in examples:
    assert fleets_sorted(target, position, speed) == expected
print("sắp xếp + pre OK")

sắp xếp + pre OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        idx = sorted(range(len(position)), key=lambda i: position[i])
        ans = pre = 0
        for i in idx[::-1]:
            t = (target - position[i]) / speed[i]
            if t > pre:
                ans += 1
                pre = t
        return ans

sol_sorted = Solution().carFleet
print([sol_sorted(t, p, s) for t, p, s, _ in examples])

[3, 1, 1]


**Độ phức tạp (Cách 2):** thời gian `O(n log n)` (sắp xếp), bộ nhớ `O(n)` (mảng chỉ số).

---

### 7.2. Cách 3 — So sánh bằng số nguyên (nhân chéo)

**Ý tưởng khác.** Giữ `pre` dưới dạng **phân số** `(num, den)` (`num = target - p`, `den = speed`) và so `t > pre` bằng nhân chéo: `num * pre_den > pre_num * den`. Không có phép chia nên **không có sai số**.

**Giải thích từng đoạn.** Khởi tạo `pre = (0, 1)` (thời gian 0). Xe có `num = target - p > 0` (vì `p < target`) nên luôn `> 0`, xe đầu tiên là đoàn mới.

In [12]:
def fleets_exact(target: int, position: List[int], speed: List[int]) -> int:
    idx = sorted(range(len(position)), key=lambda i: position[i], reverse=True)
    ans = 0
    pre_num, pre_den = 0, 1
    for i in idx:
        num, den = target - position[i], speed[i]
        if num * pre_den > pre_num * den:            # num/den > pre_num/pre_den
            ans += 1
            pre_num, pre_den = num, den
    return ans

for target, position, speed, expected in examples:
    assert fleets_exact(target, position, speed) == expected
print("nhân chéo (số nguyên) OK")

nhân chéo (số nguyên) OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        idx = sorted(range(len(position)), key=lambda i: position[i], reverse=True)
        ans = 0
        pre_num, pre_den = 0, 1
        for i in idx:
            num, den = target - position[i], speed[i]
            if num * pre_den > pre_num * den:
                ans += 1
                pre_num, pre_den = num, den
        return ans

sol_exact = Solution().carFleet
print([sol_exact(t, p, s) for t, p, s, _ in examples])

[3, 1, 1]


**Độ phức tạp (Cách 3):** thời gian `O(n log n)`, bộ nhớ `O(n)`; mọi phép tính nguyên `<= 10^12`.

---

### 7.3. Cách 4 — Ngăn xếp đơn điệu

**Ý tưởng khác.** Duyệt xe **từ phía sau ra phía trước** (vị trí tăng dần). Ngăn xếp lưu **thời gian về đích** của các đoàn hiện có (từ đoàn sau tới đoàn trước). Khi gặp xe mới với thời gian `t` (xe này đứng **trước** mọi đoàn đang có): mọi đoàn trong ngăn xếp có thời gian `<= t` sẽ bị xe này **chặn lại** và nhập vào đoàn của xe mới: `pop` chúng. Sau đó `push(t)`. Đáp án là kích thước ngăn xếp.

**Giải thích từng đoạn.**

- Sắp tăng dần theo vị trí.
- `while stack and stack[-1] <= t: stack.pop()`; `stack.append(t)`.
- Các phần tử còn lại trong ngăn xếp giảm dần từ đáy lên đỉnh? Đáy là đoàn xa nhất phía sau (thời gian lớn), đỉnh là đoàn gần đích nhất (thời gian nhỏ hơn): đúng là các đoàn không nhập nhau.

In [14]:
def fleets_stack(target: int, position: List[int], speed: List[int]) -> int:
    cars = sorted(zip(position, speed))                  # tăng dần theo vị trí (xe sau -> xe trước)
    stack = []
    for p, s in cars:
        t = Fraction(target - p, s)
        while stack and stack[-1] <= t:
            stack.pop()
        stack.append(t)
    return len(stack)

for target, position, speed, expected in examples:
    assert fleets_stack(target, position, speed) == expected
print("ngăn xếp đơn điệu OK")

ngăn xếp đơn điệu OK


**Code nộp LeetCode (Cách 4):** so sánh bằng nhân chéo để tránh số thực (lưu cặp `(num, den)`).

In [15]:
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        stack = []                                       # mỗi phần tử: (num, den) = thời gian num/den
        for p, s in sorted(zip(position, speed)):
            num, den = target - p, s
            while stack and stack[-1][0] * den <= num * stack[-1][1]:     # stack[-1] <= num/den
                stack.pop()
            stack.append((num, den))
        return len(stack)

sol_stack = Solution().carFleet
print([sol_stack(t, p, s) for t, p, s, _ in examples])

[3, 1, 1]


**Độ phức tạp (Cách 4):** thời gian `O(n log n)` (sắp xếp; vòng lặp `O(n)` vì mỗi phần tử vào/ra ngăn xếp một lần), bộ nhớ `O(n)`.

---

### 7.4. Cách 5 — Mô phỏng sự kiện đuổi kịp bằng `Fraction`

**Ý tưởng khác hẳn.** Mô phỏng đúng như mô tả vật lý, không dùng "quy tắc thời gian về đích". Các xe xếp theo thứ tự phía trước -> phía sau. Lặp:

1. Với mỗi cặp **kề nhau** (xe trước `F`, xe sau `B`) mà `speed_B > speed_F`, tính thời điểm đuổi kịp `dt = (pos_F - pos_B) / (speed_B - speed_F)` và vị trí gặp `pos_F + speed_F * dt`.
2. Lấy sự kiện **sớm nhất** có vị trí gặp `<= target` (gặp đúng tại đích vẫn nhập đoàn).
3. Tiến mọi xe thêm `dt` (cập nhật vị trí), **nhập** `B` vào đoàn của `F` (xóa `B` khỏi danh sách; đoàn chạy với tốc độ của `F`).
4. Dừng khi không còn sự kiện hợp lệ; số xe còn lại là số đoàn.

Dùng `Fraction` nên chính xác tuyệt đối. Đây cũng là **oracle độc lập**.

In [16]:
def simulate(target: int, position: List[int], speed: List[int]):
    """Trả về danh sách các đoàn (mỗi đoàn là danh sách chỉ số xe), mô phỏng sự kiện chính xác."""
    order = sorted(range(len(position)), key=lambda i: position[i], reverse=True)    # trước -> sau
    cars = [{"pos": Fraction(position[i]), "spd": Fraction(speed[i]), "members": [i]} for i in order]
    while True:
        best = None
        for k in range(len(cars) - 1):
            F, B = cars[k], cars[k + 1]
            if B["spd"] > F["spd"]:
                dt = (F["pos"] - B["pos"]) / (B["spd"] - F["spd"])
                meet = F["pos"] + F["spd"] * dt
                if meet <= target and (best is None or dt < best[0]):
                    best = (dt, k)
        if best is None:
            break
        dt, k = best
        for c in cars:
            c["pos"] += c["spd"] * dt
        cars[k]["members"] += cars[k + 1]["members"]
        del cars[k + 1]
    return [sorted(c["members"]) for c in cars]

def fleets_simulate(target: int, position: List[int], speed: List[int]) -> int:
    return len(simulate(target, position, speed))

for target, position, speed, expected in examples:
    groups = simulate(target, position, speed)
    print(position, speed, "->", groups)
    assert len(groups) == expected

[10, 8, 0, 5, 3] [2, 4, 1, 1, 3] -> [[0, 1], [3, 4], [2]]
[3] [3] -> [[0]]
[0, 2, 4] [4, 2, 1] -> [[0, 1, 2]]


**Code nộp LeetCode (Cách 5):** mô phỏng `O(n^2)` (mỗi sự kiện duyệt mọi cặp kề nhau, tối đa `n` sự kiện), chậm cho `n = 10^5`; dùng làm oracle và để hiểu bản chất, **không** nộp.

**Độ phức tạp (Cách 5):** thời gian `O(n^2)` (cho `n` xe: tối đa `n - 1` sự kiện, mỗi sự kiện `O(n)`), bộ nhớ `O(n)`. Với `Fraction` hằng số lớn hơn nữa.

## 8. TRACE từng bước

### 8.1. Cách 2 trên ví dụ 1

Đã tính ở mục 4: các xe theo vị trí giảm dần `10, 8, 5, 3, 0` có `t = 1, 1, 7, 3, 12` và `pre` đi `0 -> 1 -> 1 -> 7 -> 7 -> 12`; số lần `pre` tăng ngặt là 3.

Hàm `trace_sorted` in cả `idx` đã sắp xếp theo vị trí.

In [17]:
def trace_sorted(target, position, speed):
    n = len(position)
    idx = sorted(range(n), key=lambda i: position[i])
    print("idx (tăng dần theo vị trí):", idx, "-> vị trí:", [position[i] for i in idx])
    ans, pre = 0, 0.0
    for i in idx[::-1]:
        t = (target - position[i]) / speed[i]
        if t > pre:
            ans += 1
            pre = t
            print(f"  xe {i} (vị trí {position[i]}, tốc độ {speed[i]}): t = {t:.3f} > pre -> đoàn #{ans}, pre = {pre:.3f}")
        else:
            print(f"  xe {i} (vị trí {position[i]}, tốc độ {speed[i]}): t = {t:.3f} <= pre = {pre:.3f} -> nhập đoàn")
    return ans

assert trace_sorted(*examples[0][:3]) == 3
print()
assert trace_sorted(*examples[2][:3]) == 1

idx (tăng dần theo vị trí): [2, 4, 3, 1, 0] -> vị trí: [0, 3, 5, 8, 10]
  xe 0 (vị trí 10, tốc độ 2): t = 1.000 > pre -> đoàn #1, pre = 1.000
  xe 1 (vị trí 8, tốc độ 4): t = 1.000 <= pre = 1.000 -> nhập đoàn
  xe 3 (vị trí 5, tốc độ 1): t = 7.000 > pre -> đoàn #2, pre = 7.000
  xe 4 (vị trí 3, tốc độ 3): t = 3.000 <= pre = 7.000 -> nhập đoàn
  xe 2 (vị trí 0, tốc độ 1): t = 12.000 > pre -> đoàn #3, pre = 12.000

idx (tăng dần theo vị trí): [0, 1, 2] -> vị trí: [0, 2, 4]
  xe 2 (vị trí 4, tốc độ 1): t = 96.000 > pre -> đoàn #1, pre = 96.000
  xe 1 (vị trí 2, tốc độ 2): t = 49.000 <= pre = 96.000 -> nhập đoàn
  xe 0 (vị trí 0, tốc độ 4): t = 25.000 <= pre = 96.000 -> nhập đoàn


### 8.2. Cách 4 (ngăn xếp) trên ví dụ 1

Sắp tăng dần theo vị trí: `(0,1)`, `(3,3)`, `(5,1)`, `(8,4)`, `(10,2)` với `t = 12, 3, 7, 1, 1`.

| xe `(p, s)` | `t` | thao tác | ngăn xếp (thời gian) |
|---|---|---|---|
| (0, 1) | 12 | đẩy | `[12]` |
| (3, 3) | 3 | `12 <= 3`? không; đẩy | `[12, 3]` |
| (5, 1) | 7 | `3 <= 7` pop; `12 <= 7`? không; đẩy | `[12, 7]` |
| (8, 4) | 1 | `7 <= 1`? không; đẩy | `[12, 7, 1]` |
| (10, 2) | 1 | `1 <= 1` pop; `7 <= 1`? không; đẩy | `[12, 7, 1]` |

Kích thước 3.

In [18]:
def trace_stack(target, position, speed):
    stack = []
    for p, s in sorted(zip(position, speed)):
        t = Fraction(target - p, s)
        popped = []
        while stack and stack[-1] <= t:
            popped.append(stack.pop())
        stack.append(t)
        print(f"xe ({p:2}, {s}): t = {str(t):>3}; pop {[str(x) for x in popped]} ; ngăn xếp = {[str(x) for x in stack]}")
    return len(stack)

assert trace_stack(*examples[0][:3]) == 3

xe ( 0, 1): t =  12; pop [] ; ngăn xếp = ['12']
xe ( 3, 3): t =   3; pop [] ; ngăn xếp = ['12', '3']
xe ( 5, 1): t =   7; pop ['3'] ; ngăn xếp = ['12', '7']
xe ( 8, 4): t =   1; pop [] ; ngăn xếp = ['12', '7', '1']
xe (10, 2): t =   1; pop ['1'] ; ngăn xếp = ['12', '7', '1']


### 8.3. Cách 5: các sự kiện đuổi kịp ở ví dụ 3 (`target = 100`, `position = [0, 2, 4]`, `speed = [4, 2, 1]`)

Xe trước là xe ở 4 (tốc độ 1). Xe ở 2 (tốc độ 2) đuổi kịp lúc `dt = (4 - 2) / (2 - 1) = 2`, vị trí 6; xe ở 0 (tốc độ 4) đuổi kịp xe ở 2 lúc `dt = 2/2 = 1`, vị trí 4. Sự kiện sớm nhất là `dt = 1`: nhập xe 0 và xe 1 (vị trí 4, tốc độ 2); rồi đoàn này đuổi kịp xe ở 5 (lúc đó xe trước ở 5): ...

In [19]:
def trace_simulate(target, position, speed):
    order = sorted(range(len(position)), key=lambda i: position[i], reverse=True)
    cars = [{"pos": Fraction(position[i]), "spd": Fraction(speed[i]), "members": [i]} for i in order]
    T = Fraction(0)
    print("ban đầu (trước -> sau):", [(str(c["pos"]), str(c["spd"]), c["members"]) for c in cars])
    while True:
        best = None
        for k in range(len(cars) - 1):
            F, B = cars[k], cars[k + 1]
            if B["spd"] > F["spd"]:
                dt = (F["pos"] - B["pos"]) / (B["spd"] - F["spd"])
                meet = F["pos"] + F["spd"] * dt
                if meet <= target and (best is None or dt < best[0]):
                    best = (dt, k, meet)
        if best is None:
            break
        dt, k, meet = best
        T += dt
        for c in cars:
            c["pos"] += c["spd"] * dt
        print(f"t = {T}: xe {cars[k + 1]['members']} đuổi kịp {cars[k]['members']} tại vị trí {meet} -> nhập đoàn")
        cars[k]["members"] += cars[k + 1]["members"]
        del cars[k + 1]
    print("kết quả:", [sorted(c["members"]) for c in cars])
    return len(cars)

assert trace_simulate(*examples[2][:3]) == 1
print()
assert trace_simulate(*examples[0][:3]) == 3

ban đầu (trước -> sau): [('4', '1', [2]), ('2', '2', [1]), ('0', '4', [0])]
t = 1: xe [0] đuổi kịp [1] tại vị trí 4 -> nhập đoàn
t = 2: xe [1, 0] đuổi kịp [2] tại vị trí 6 -> nhập đoàn
kết quả: [[0, 1, 2]]

ban đầu (trước -> sau): [('10', '2', [0]), ('8', '4', [1]), ('5', '1', [3]), ('3', '3', [4]), ('0', '1', [2])]
t = 1: xe [1] đuổi kịp [0] tại vị trí 12 -> nhập đoàn
t = 1: xe [4] đuổi kịp [3] tại vị trí 6 -> nhập đoàn
kết quả: [[0, 1], [3, 4], [2]]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (sắp xếp + theo dõi thời gian lớn nhất): ngắn, `O(n log n)`, và an toàn với số thực dưới ràng buộc của đề (chứng minh ở mục 10). Nếu muốn tránh số thực hoàn toàn, dùng **Cách 3** (nhân chéo).

```python
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        idx = sorted(range(len(position)), key=lambda i: position[i])
        ans = pre = 0
        for i in idx[::-1]:
            t = (target - position[i]) / speed[i]
            if t > pre:
                ans += 1
                pre = t
        return ans
```

In [20]:
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        idx = sorted(range(len(position)), key=lambda i: position[i])
        ans = pre = 0
        for i in idx[::-1]:
            t = (target - position[i]) / speed[i]
            if t > pre:
                ans += 1
                pre = t
        return ans

sol = Solution()
for target, position, speed, expected in examples:
    got = sol.carFleet(target, position, speed)
    print(position, speed, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[10, 8, 0, 5, 3] [2, 4, 1, 1, 3] -> 3 OK
[3] [3] -> 1 OK
[0, 2, 4] [4, 2, 1] -> 1 OK


## 10. Vì sao đúng

**(a) Chỉ cần thời gian tự do.** Xét hai xe kề nhau trong thứ tự vị trí: `F` ở trước, `B` ở sau. Nếu `B` không đuổi kịp `F` trước đích thì cả hai tới đích độc lập; nếu `B` đuổi kịp `F` (hoặc `F`-đoàn) thì từ lúc đó `B` đi cùng với `F`. Tốc độ của đoàn luôn là tốc độ nhỏ nhất của đoàn (xe đứng đầu bị chặn bởi chính nó, xe phía sau bị chặn lại), và cuối cùng thời gian về đích của đoàn = **thời gian tự do lớn nhất** trong các xe của đoàn (xe chậm nhất trong "ảnh hưởng" tới đích quyết định). Cụ thể: một xe `B` đuổi kịp một xe/đoàn phía trước trước (hoặc tại) đích **khi và chỉ khi** thời gian tự do của `B` **không lớn hơn** thời gian về đích (thực sự) của xe/đoàn phía trước. Thời gian về đích thực sự của đoàn đứng ngay trước `B` là thời gian tự do lớn nhất `pre` trong các xe phía trước (xe tự do chậm nhất quyết định, và các xe còn lại bị nó chặn).

**(b) Hệ quả.** Duyệt từ trước ra sau: xe có `t > pre` không bao giờ bị chặn (đoàn mới, `pre = t`); xe có `t <= pre` bị chặn (nhập đoàn, `pre` giữ nguyên). Số đoàn = số **kỷ lục mới** (strict) của `pre`.

**(c) Dấu `>` hay `>=`.** Nếu `t == pre` thì xe đuổi kịp đoàn **đúng tại đích**; đề nói vẫn tính cùng đoàn, nên `t == pre` **không** tạo đoàn mới: điều kiện đoàn mới là `t > pre` (ngặt).

**(d) Số thực an toàn.** Mọi `t = a / b` với `a = target - position < 10^6`, `1 <= b <= 10^6`, và hai `t` khác nhau `a/b - c/d` có chênh lệch tuyệt đối ít nhất `1/(bd)`. Chia cho `t <= a / b` ta được chênh lệch **tương đối** `>= 1 / (a d) >= 10^-12` (vì `a, d <= 10^6`). Sai số tương đối của phép chia số thực kép đúng làm tròn là `<= 2^-53 ≈ 1.1 * 10^-16`, nhỏ hơn rất nhiều, nên thứ tự (và đẳng thức) của các `t` được số thực giữ nguyên; hai `t` bằng nhau cho đúng cùng số thực (phép chia được làm tròn đúng của cùng một giá trị thực). Ta kiểm bằng cách so Cách 2 với Cách 3 (chính xác) trên nhiều input.

**(e) Cách 4.** Duyệt từ sau ra trước: một xe mới đứng **trước** mọi đoàn đang có trong ngăn xếp; nếu thời gian của đoàn `<= t` của xe mới thì đoàn đó đuổi kịp xe mới không muộn hơn đích nên nhập vào. Đoàn nào có thời gian `> t` thì không bị ảnh hưởng và vẫn đứng sau xe mới. Phần còn lại của ngăn xếp có thời gian giảm dần từ đáy (xa nhất phía sau) tới đỉnh (gần đích nhất), tức là các đoàn **riêng biệt**.

**(f) Cách 5.** Là mô phỏng trực tiếp định nghĩa nên không dựa vào quy tắc (a)-(c); khớp với các cách kia trên mọi input chứng tỏ quy tắc đúng.

**Kiểm chứng bằng máy:** so mọi cách với oracle mô phỏng sự kiện chính xác trên **mọi** cấu hình nhỏ (`target <= 6`, mọi tập vị trí, tốc độ `1..3`) và trên nhiều input ngẫu nhiên; so Cách 2 (số thực) với Cách 3 (số nguyên) trên input lớn nhất theo đề.

In [21]:
from itertools import combinations, product

solutions = {
    "pairs": fleets_pairs,
    "sorted": fleets_sorted,
    "exact": fleets_exact,
    "stack": fleets_stack,
    "simulate": fleets_simulate,
    "sol_sorted": lambda t, p, s: sol_sorted(t, p, s),
    "sol_exact": lambda t, p, s: sol_exact(t, p, s),
    "sol_stack": lambda t, p, s: sol_stack(t, p, s),
    "final": lambda t, p, s: Solution().carFleet(t, p, s),
}

checked = 0
for target in range(1, 7):
    for n in range(1, min(target, 4) + 1):
        for pos in combinations(range(target), n):               # vị trí khác nhau trong [0, target)
            for spd in product(range(1, 4), repeat=n):
                p, s = list(pos), list(spd)
                ref = len(simulate(target, p, s))
                for name, fn in solutions.items():
                    assert fn(target, p, s) == ref, (name, target, p, s, fn(target, p, s), ref)
                checked += 1
print(f"mọi cấu hình nhỏ (target <= 6, n <= 4, tốc độ 1..3): {checked} ca x {len(solutions)} cài đặt khớp mô phỏng chính xác")

mọi cấu hình nhỏ (target <= 6, n <= 4, tốc độ 1..3): 3024 ca x 9 cài đặt khớp mô phỏng chính xác


## 11. Time / Space Complexity

Gọi `n = len(position)` (`<= 10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. So từng cặp | `O(n^2)` | `O(n)` |
| 2. Sắp xếp + `pre` | `O(n log n)` | `O(n)` |
| 3. Nhân chéo (số nguyên) | `O(n log n)` | `O(n)` |
| 4. Ngăn xếp đơn điệu | `O(n log n)` | `O(n)` |
| 5. Mô phỏng sự kiện | `O(n^2)` | `O(n)` |

Với `n = 10^5`, `O(n log n)` chạy dưới nửa giây. Đo (bỏ qua các cách `O(n^2)`):

In [22]:
rng = random.Random(853)
big = []
for _ in range(5):
    big.append(make_cars(100000, 10 ** 6, rng))
big.append((10 ** 6, list(range(100000)), [1] * 100000))                      # mọi tốc độ bằng nhau: mỗi xe một đoàn
big.append((10 ** 6, list(range(100000)), list(range(100000, 0, -1))))        # xe phía sau nhanh hơn: nhập hết

ref = None
for name in ["sorted", "exact", "stack"]:
    t0 = time.perf_counter()
    res = [solutions[name](*c) for c in big]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:7} {len(big)} test n=10^5: {dt:9.1f} ms")
print("số đoàn:", ref)

sorted  7 test n=10^5:     429.1 ms


exact   7 test n=10^5:     505.4 ms


stack   7 test n=10^5:    1942.9 ms
số đoàn: [27, 18, 21, 22, 20, 100000, 1]


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle mô phỏng, và test ngẫu nhiên có seed (đúng ràng buộc: vị trí khác nhau trong `[0, target)`, tốc độ dương).

In [23]:
# 12.1 ví dụ đề
for target, position, speed, expected in examples:
    for name, fn in solutions.items():
        assert fn(target, position, speed) == expected, (name, position, speed)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (10, [0], [1], 1),                                   # một xe
    (10, [0, 5], [1, 1], 2),                             # cùng tốc độ: không bao giờ gặp
    (10, [0, 5], [2, 1], 1),                             # xe sau nhanh hơn, về sớm hơn nên bị chặn: t_sau = 5 <= t_trước = 5 (bằng nhau tại đích)
    (10, [0, 5], [3, 1], 1),                             # xe sau nhanh hơn rõ ràng
    (10, [0, 5], [1, 2], 2),                             # xe trước nhanh hơn: không gặp
    (10, [0, 5], [10, 1], 1),                            # xe sau cực nhanh
    (10, [8, 0], [1, 1], 2),
    (12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3], 3),          # ví dụ 1 (thứ tự không sắp xếp sẵn)
    (100, [0, 2, 4], [4, 2, 1], 1),                      # ví dụ 3
    (10, [9, 7, 5, 3, 1], [1, 1, 1, 1, 1], 5),           # cùng tốc độ: mỗi xe một đoàn
    (10, [1, 3, 5, 7, 9], [9, 7, 5, 3, 1], 1),           # xe phía sau luôn nhanh hơn xe phía trước
    (10, [1, 3, 5, 7, 9], [1, 2, 3, 4, 5], 5),           # xe phía trước luôn nhanh hơn: không ai gặp ai
    (10 ** 6, [0], [10 ** 6], 1),                        # giá trị lớn nhất
    (10 ** 6, [999999, 0], [1, 10 ** 6], 1),             # t = 1 cho cả hai xe: gặp đúng tại đích -> một đoàn
]
for target, position, speed, expected in edge:
    assert len(simulate(target, position, speed)) == expected, ("oracle khác kỳ vọng", target, position, speed, len(simulate(target, position, speed)))
    for name, fn in solutions.items():
        assert fn(target, position, speed) == expected, (name, target, position, speed, fn(target, position, speed), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [24]:
# 12.3 phản ví dụ: các cách sai
def wrong_no_sort(target, position, speed):               # quên sắp xếp theo vị trí
    ans = pre = 0
    for p, s in zip(position, speed):
        t = (target - p) / s
        if t > pre:
            ans += 1
            pre = t
    return ans

def wrong_sort_separately(target, position, speed):       # sắp xếp riêng position và speed
    pos, spd = sorted(position), sorted(speed)
    ans = pre = 0
    for p, s in reversed(list(zip(pos, spd))):
        t = (target - p) / s
        if t > pre:
            ans += 1
            pre = t
    return ans

def wrong_ge(target, position, speed):                    # dùng >= : đoàn gặp đúng tại đích bị đếm là hai
    ans = pre = 0
    for p, s in sorted(zip(position, speed), reverse=True):
        t = (target - p) / s
        if t >= pre:
            ans += 1
            pre = t
    return ans

def wrong_floor_div(target, position, speed):             # chia nguyên
    ans = pre = 0
    for p, s in sorted(zip(position, speed), reverse=True):
        t = (target - p) // s
        if t > pre:
            ans += 1
            pre = t
    return ans

def wrong_ascending(target, position, speed):             # duyệt từ sau ra trước nhưng đếm "kỷ lục lớn nhất" giống bản trước
    ans = pre = 0
    for p, s in sorted(zip(position, speed)):
        t = (target - p) / s
        if t > pre:
            ans += 1
            pre = t
    return ans

wrongs = [
    ("quên sắp xếp theo vị trí", wrong_no_sort, 12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]),
    ("sắp xếp riêng position và speed", wrong_sort_separately, 12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]),
    ("dùng >= thay vì >", wrong_ge, 12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]),
    ("chia nguyên //", wrong_floor_div, 10, [0, 4], [3, 2]),
    ("duyệt ngược chiều", wrong_ascending, 12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]),
]
for name, fn, target, position, speed in wrongs:
    truth = len(simulate(target, position, speed))
    print(f"{name:34} sai = {fn(target, position, speed)}, đúng = {truth}")
    assert fn(target, position, speed) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

quên sắp xếp theo vị trí           sai = 2, đúng = 3
sắp xếp riêng position và speed    sai = 5, đúng = 3
dùng >= thay vì >                  sai = 4, đúng = 3
chia nguyên //                     sai = 1, đúng = 2
duyệt ngược chiều                  sai = 1, đúng = 3
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [25]:
# 12.4 ngẫu nhiên nhỏ so với oracle mô phỏng (target tới 30, n tới 8)
rng = random.Random(2853)
for _ in range(600):
    target = rng.randint(2, 30)
    n = rng.randint(1, min(8, target))
    position = rng.sample(range(target), n)
    speed = [rng.randint(1, rng.choice([2, 5, 20])) for _ in range(n)]
    ref = len(simulate(target, position, speed))
    for name, fn in solutions.items():
        assert fn(target, position, speed) == ref, (name, target, position, speed)
print("12.4 ngẫu nhiên nhỏ so với oracle mô phỏng: OK")

# 12.5 ngẫu nhiên lớn: Cách 2 (số thực) khớp Cách 3 (số nguyên chính xác) và Cách 4; oracle mô phỏng khi n <= 60
rng = random.Random(77)
for _ in range(60):
    n = rng.randint(1000, 100000)
    target, position, speed = make_cars(n, 10 ** 6, rng, max_speed=rng.choice([10, 1000, 10 ** 6]))
    a, b, c = fleets_sorted(target, position, speed), fleets_exact(target, position, speed), fleets_stack(target, position, speed)
    assert a == b == c, (n, a, b, c)
for _ in range(40):
    n = rng.randint(2, 60)
    target = rng.choice([100, 10 ** 4, 10 ** 6])
    position = rng.sample(range(target), n)
    speed = [rng.randint(1, rng.choice([5, 100, 10 ** 6])) for _ in range(n)]
    assert fleets_exact(target, position, speed) == len(simulate(target, position, speed))
print("12.5 ngẫu nhiên lớn và trung bình: OK")

# 12.6 các giá trị thời gian gần nhau nhất vẫn phân biệt được bằng số thực (kiểm tra chặn của mục 10(d))
rng = random.Random(6)
for _ in range(20000):
    a, c = rng.randint(1, 10 ** 6 - 1), rng.randint(1, 10 ** 6 - 1)
    b, d = rng.randint(1, 10 ** 6), rng.randint(1, 10 ** 6)
    if a * d == c * b:
        assert a / b == c / d                              # bằng nhau thật -> số thực cũng bằng nhau
    else:
        assert (a / b < c / d) == (a * d < c * b)           # thứ tự số thực khớp thứ tự chính xác
print("12.6 thứ tự và đẳng thức của số thực khớp số chính xác trên 20000 cặp ngẫu nhiên: OK")

12.4 ngẫu nhiên nhỏ so với oracle mô phỏng: OK


12.5 ngẫu nhiên lớn và trung bình: OK
12.6 thứ tự và đẳng thức của số thực khớp số chính xác trên 20000 cặp ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp theo vị trí

`position` không nhất thiết đã sắp xếp. Quy tắc "so với `pre`" chỉ đúng khi duyệt xe **theo thứ tự vị trí**.

In [26]:
target, position, speed = 12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]
print("sai (không sắp xếp):", wrong_no_sort(target, position, speed), "| đúng:", Solution().carFleet(target, position, speed))
assert wrong_no_sort(target, position, speed) != 3

sai (không sắp xếp): 2 | đúng: 3


Vì sao sai: duyệt theo thứ tự cho sẵn, `pre` không phải thời gian của xe đứng ngay trước. **Đúng:** `sorted(range(n), key=lambda i: position[i])`.

### Lỗi 2 — Sắp xếp riêng `position` và `speed`

Mỗi xe là một **cặp**; sắp xếp riêng làm tốc độ gán nhầm xe.

In [27]:
print("sai (sort riêng):", wrong_sort_separately(target, position, speed), "| đúng:", Solution().carFleet(target, position, speed))
assert wrong_sort_separately(target, position, speed) != 3

sai (sort riêng): 5 | đúng: 3


Vì sao sai: `sorted(speed)` mất tương ứng với `position`. **Đúng:** sắp xếp cặp (`zip`) hoặc sắp xếp chỉ số.

### Lỗi 3 — Dùng `>=` thay vì `>`

Hai xe gặp nhau **đúng tại đích** vẫn là một đoàn (đề nói rõ).

In [28]:
print("sai (>=):", wrong_ge(target, position, speed), "| đúng:", Solution().carFleet(target, position, speed))
assert wrong_ge(target, position, speed) == 4 and Solution().carFleet(target, position, speed) == 3

sai (>=): 4 | đúng: 3


Vì sao sai: xe ở 10 (tốc độ 2) và xe ở 8 (tốc độ 4) cùng về đích lúc `t = 1`; `>=` đếm thành hai đoàn. **Đúng:** `t > pre`.

### Lỗi 4 — Chia nguyên `//`

Thời gian là số thực (phân số); chia nguyên làm tròn xuống và làm mất khác biệt.

In [29]:
t2, p2, s2 = 10, [0, 4], [3, 2]           # t = 10/3 ≈ 3,33 và 6/2 = 3: khác nhau nhưng // cho cùng 3
print("sai (//):", wrong_floor_div(t2, p2, s2), "| đúng:", Solution().carFleet(t2, p2, s2))
assert wrong_floor_div(t2, p2, s2) != Solution().carFleet(t2, p2, s2)

sai (//): 1 | đúng: 2


Vì sao sai: `10 // 3 = 3` và `6 // 2 = 3` bằng nhau nên bị coi là cùng đoàn, dù `10/3 > 3` (xe sau về muộn hơn, không đuổi kịp). **Đúng:** chia thực hoặc nhân chéo.

### Lỗi 5 — Duyệt ngược chiều

Quy tắc `pre` áp dụng khi duyệt **từ trước ra sau** (vị trí giảm dần). Duyệt từ sau ra trước với cùng điều kiện cho kết quả khác.

In [30]:
print("sai (duyệt tăng dần):", wrong_ascending(target, position, speed), "| đúng:", Solution().carFleet(target, position, speed))
assert wrong_ascending(target, position, speed) != 3

sai (duyệt tăng dần): 1 | đúng: 3


Vì sao sai: xe phía sau chưa biết đoàn phía trước. Nếu muốn duyệt từ sau ra trước phải dùng **ngăn xếp** (Cách 4) chứ không phải một `pre` duy nhất.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** các đối tượng trên một đường thẳng **không vượt nhau**, "bị chặn bởi đối tượng phía trước" -> sắp xếp theo vị trí và so với **đại lượng quyết định** (ở đây thời gian về đích).
- **Chuyển bài mô phỏng thành so sánh:** thay vì mô phỏng, tìm một đại lượng bất biến (`t_i`) rồi đếm "kỷ lục" hoặc dùng ngăn xếp đơn điệu.
- **Ghép cặp khi sắp xếp:** sắp xếp chỉ số hoặc `zip`, không sắp xếp từng mảng riêng.
- **So sánh `>` hay `>=`:** đọc kỹ điều kiện biên (đề nói gặp tại đích vẫn một đoàn).
- **Số thực vs số nguyên:** khi có thể, so sánh phân số bằng nhân chéo; kiểm tra bằng `Fraction`.
- **Oracle mô phỏng sự kiện** bằng `Fraction` là cách chắc chắn để kiểm các quy tắc "ngắn gọn".
- Cùng họ: Daily Temperatures (739), Next Greater Element, Asteroid Collision (735), Car Fleet II (1776).
- Test nên có: một xe, mọi tốc độ bằng nhau, xe sau nhanh hơn hẳn, gặp đúng tại đích, `n = 10^5`.

## 15. Biến thể

1. **Trả về các đoàn** (danh sách chỉ số xe trong mỗi đoàn): bài tập 5; kiểm tra bằng oracle mô phỏng.
2. **Thời điểm đoàn cuối cùng về đích** = `max(t_i)` (thời gian lớn nhất).
3. **Car Fleet II (1776):** với mỗi xe, thời điểm nó va vào xe phía trước (ngăn xếp + tính thời điểm): khó hơn.
4. Bài liên quan: Asteroid Collision (735), Daily Temperatures (739).

Dưới đây là code cho biến thể 1 (nhóm các xe theo đoàn bằng sắp xếp) và 2.

In [31]:
def fleet_groups(target: int, position: List[int], speed: List[int]) -> List[List[int]]:
    order = sorted(range(len(position)), key=lambda i: position[i], reverse=True)
    groups = []
    pre_num, pre_den = 0, 1
    for i in order:
        num, den = target - position[i], speed[i]
        if num * pre_den > pre_num * den:
            groups.append([i])
            pre_num, pre_den = num, den
        else:
            groups[-1].append(i)
    return sorted(sorted(g) for g in groups)

def last_arrival(target: int, position: List[int], speed: List[int]) -> Fraction:
    return max(Fraction(target - p, s) for p, s in zip(position, speed))

rng = random.Random(15)
for _ in range(400):
    target = rng.randint(2, 40)
    n = rng.randint(1, min(8, target))
    position = rng.sample(range(target), n)
    speed = [rng.randint(1, 6) for _ in range(n)]
    assert fleet_groups(target, position, speed) == sorted(simulate(target, position, speed)), (target, position, speed)
print(fleet_groups(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]), "| thời điểm cuối:", last_arrival(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))
print("Biến thể: OK")

[[0, 1], [2], [3, 4]] | thời điểm cuối: 12
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | Xe là đầu đoàn nếu không xe trước nào có `t >=` | `O(n^2)` | `O(n)` | Hiểu quy tắc; `n` nhỏ |
| 2. Sắp xếp + `pre` | Kỷ lục `t` khi duyệt từ trước ra sau | `O(n log n)` | `O(n)` | **Cách nộp chính** |
| 3. Nhân chéo | Cùng Cách 2, số nguyên | `O(n log n)` | `O(n)` | Khi không muốn số thực |
| 4. Ngăn xếp đơn điệu | Duyệt từ sau ra trước, pop đoàn `t <=` | `O(n log n)` | `O(n)` | Khi mở rộng (Car Fleet II) |
| 5. Mô phỏng sự kiện | Đuổi kịp theo thời gian, `Fraction` | `O(n^2)` | `O(n)` | Oracle, hiểu vật lý |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(sorted(zip([3, 1, 2], [30, 10, 20])))
print([(10 - p) / s for p, s in [(4, 2), (0, 5)]])
print(Solution().carFleet(10, [0, 5], [3, 1]))
```

In [32]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `target = 20`, `position = [2, 6, 10, 15]`, `speed = [4, 3, 2, 1]`: điền thời gian `t` của từng xe (đúng thứ tự cho sẵn) và số đoàn.

In [33]:
# Bài 2: điền kết quả tính tay
hand_times = None       # list 4 số (có thể dùng Fraction hoặc số thực)
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách sắp xếp + theo dõi thời gian lớn nhất.

In [34]:
def exercise_3_sorted(target: int, position: List[int], speed: List[int]) -> int:
    # TODO 1: idx = các chỉ số xe, sắp xếp tăng dần theo position
    ans = pre = 0
    # TODO 2: duyệt idx từ cuối về đầu (xe gần đích nhất trước)
    #     TODO 3: t = (target - position[i]) / speed[i]
    #     TODO 4: nếu t > pre thì ans += 1 và pre = t
    return None


# assert exercise_3_sorted(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
# assert exercise_3_sorted(10, [3], [3]) == 1
# assert exercise_3_sorted(100, [0, 2, 4], [4, 2, 1]) == 1

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_stack(target, position, speed)` bằng ngăn xếp đơn điệu (duyệt từ xe phía sau ra phía trước).

In [35]:
def exercise_4_stack(target: int, position: List[int], speed: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_stack(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
# assert exercise_4_stack(10, [3], [3]) == 1
# assert exercise_4_stack(100, [0, 2, 4], [4, 2, 1]) == 1

### Bài 5 — Mini challenge: liệt kê các đoàn (khó)

Viết `exercise_5_groups(target, position, speed)` trả về danh sách các đoàn, mỗi đoàn là **danh sách chỉ số xe** (chỉ số trong mảng đầu vào), mỗi danh sách sắp xếp tăng dần, và danh sách các đoàn cũng sắp xếp. Kiểm tra bằng `simulate` (oracle).

```python
exercise_5_groups(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3])  -> [[0, 1], [2], [3, 4]]
```
Gợi ý: khi một xe không phải đầu đoàn mới thì thêm vào đoàn vừa tạo gần nhất.

In [36]:
def exercise_5_groups(target: int, position: List[int], speed: List[int]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_5_groups(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == [[0, 1], [2], [3, 4]]
# assert exercise_5_groups(10, [3], [3]) == [[0]]
# assert exercise_5_groups(100, [0, 2, 4], [4, 2, 1]) == [[0, 1, 2]]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `zip` ghép phần tử cùng chỉ số; `sorted` sắp theo phần tử đầu của cặp.
- Mức 2: `(10 - 4) / 2 = 3`, `(10 - 0) / 5 = 2`.
- Mức 3: `[(1, 10), (2, 20), (3, 30)]`, `[3.0, 2.0]`, `1` (xe sau `t = 10/3 ≈ 3,33 <= ` xe trước `t = 5/1 = 5`).

**Bài 2**
- Mức 1: `t = (20 - p) / s`.
- Mức 2: `(18)/4 = 4,5`; `14/3 ≈ 4,67`; `10/2 = 5`; `5/1 = 5`.
- Mức 3: `[4.5, 4.67, 5, 5]`; duyệt từ trước ra sau: xe ở 15 (`5`) -> đoàn #1; xe ở 10 (`5`, không `> 5`) nhập; xe ở 6 (`4,67`) nhập; xe ở 2 (`4,5`) nhập -> 1 đoàn.

**Bài 3**
- Mức 1: `idx = sorted(range(len(position)), key=lambda i: position[i])`.
- Mức 2: `for i in idx[::-1]:`.
- Mức 3: `t = (target - position[i]) / speed[i]`; `if t > pre: ans += 1; pre = t`.

**Bài 4**
- Mức 1: `for p, s in sorted(zip(position, speed))`.
- Mức 2: `while stack and stack[-1] <= t: stack.pop()`.
- Mức 3: `stack.append(t)`; `return len(stack)`.

**Bài 5**
- Mức 1: sắp xếp chỉ số theo `position` giảm dần.
- Mức 2: so nhân chéo với `pre`; nếu lớn hơn: tạo đoàn mới `[i]`, ngược lại `groups[-1].append(i)`.
- Mức 3: cuối cùng `sorted(sorted(g) for g in groups)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(sorted(zip([3, 1, 2], [30, 10, 20])))
print([(10 - p) / s for p, s in [(4, 2), (0, 5)]])
print(Solution().carFleet(10, [0, 5], [3, 1]))

assert sorted(zip([3, 1, 2], [30, 10, 20])) == [(1, 10), (2, 20), (3, 30)]
assert [(10 - p) / s for p, s in [(4, 2), (0, 5)]] == [3.0, 2.0]
assert Solution().carFleet(10, [0, 5], [3, 1]) == 1

[(1, 10), (2, 20), (3, 30)]
[3.0, 2.0]
1


### Đáp án Bài 2

`t = 18/4 = 4,5`; `14/3 ≈ 4,667`; `10/2 = 5`; `5/1 = 5`. Duyệt từ xe ở 15: `5 > 0` -> đoàn 1; xe ở 10: `5` không `> 5` -> nhập; xe ở 6: `4,667 <= 5` -> nhập; xe ở 2: `4,5 <= 5` -> nhập. Một đoàn.

In [38]:
hand_times_solution = [Fraction(18, 4), Fraction(14, 3), Fraction(10, 2), Fraction(5, 1)]
hand_answer_solution = 1
times = [Fraction(20 - p, s) for p, s in zip([2, 6, 10, 15], [4, 3, 2, 1])]
assert times == hand_times_solution
assert Solution().carFleet(20, [2, 6, 10, 15], [4, 3, 2, 1]) == hand_answer_solution == len(simulate(20, [2, 6, 10, 15], [4, 3, 2, 1]))
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_sorted_solution(target: int, position: List[int], speed: List[int]) -> int:
    idx = sorted(range(len(position)), key=lambda i: position[i])
    ans = pre = 0
    for i in idx[::-1]:
        t = (target - position[i]) / speed[i]
        if t > pre:
            ans += 1
            pre = t
    return ans

assert exercise_3_sorted_solution(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
assert exercise_3_sorted_solution(10, [3], [3]) == 1
assert exercise_3_sorted_solution(100, [0, 2, 4], [4, 2, 1]) == 1
rng = random.Random(3)
for _ in range(200):
    target = rng.randint(2, 30)
    n = rng.randint(1, min(7, target))
    position = rng.sample(range(target), n)
    speed = [rng.randint(1, 8) for _ in range(n)]
    assert exercise_3_sorted_solution(target, position, speed) == len(simulate(target, position, speed))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_stack_solution(target: int, position: List[int], speed: List[int]) -> int:
    stack = []
    for p, s in sorted(zip(position, speed)):
        t = Fraction(target - p, s)
        while stack and stack[-1] <= t:
            stack.pop()
        stack.append(t)
    return len(stack)

assert exercise_4_stack_solution(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
assert exercise_4_stack_solution(10, [3], [3]) == 1
assert exercise_4_stack_solution(100, [0, 2, 4], [4, 2, 1]) == 1
rng = random.Random(4)
for _ in range(200):
    target = rng.randint(2, 30)
    n = rng.randint(1, min(7, target))
    position = rng.sample(range(target), n)
    speed = [rng.randint(1, 8) for _ in range(n)]
    assert exercise_4_stack_solution(target, position, speed) == len(simulate(target, position, speed))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_groups_solution(target: int, position: List[int], speed: List[int]) -> List[List[int]]:
    order = sorted(range(len(position)), key=lambda i: position[i], reverse=True)
    groups = []
    pre_num, pre_den = 0, 1
    for i in order:
        num, den = target - position[i], speed[i]
        if num * pre_den > pre_num * den:
            groups.append([i])
            pre_num, pre_den = num, den
        else:
            groups[-1].append(i)
    return sorted(sorted(g) for g in groups)

assert exercise_5_groups_solution(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == [[0, 1], [2], [3, 4]]
assert exercise_5_groups_solution(10, [3], [3]) == [[0]]
assert exercise_5_groups_solution(100, [0, 2, 4], [4, 2, 1]) == [[0, 1, 2]]
rng = random.Random(5)
for _ in range(300):
    target = rng.randint(2, 40)
    n = rng.randint(1, min(8, target))
    position = rng.sample(range(target), n)
    speed = [rng.randint(1, 8) for _ in range(n)]
    assert exercise_5_groups_solution(target, position, speed) == sorted(simulate(target, position, speed)), (target, position, speed)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (sắp xếp theo vị trí, đếm kỷ lục thời gian)

```python
class Solution:
    def carFleet(self, target: int, position: List[int], speed: List[int]) -> int:
        idx = sorted(range(len(position)), key=lambda i: position[i])   # giữ cặp (vị trí, tốc độ)
        ans = pre = 0
        for i in idx[::-1]:                                             # từ xe gần đích nhất
            t = (target - position[i]) / speed[i]                       # thời gian tự do về đích
            if t > pre:                                                 # lớn hơn MỌI xe phía trước (ngặt)
                ans += 1
                pre = t
        return ans
```

Công thức nhớ nhanh: `t = (target - p) / s`; xe là **đầu đoàn mới** khi `t >` thời gian lớn nhất của các xe phía trước.

### Checklist trước khi submit

- [ ] Đã sắp xếp **theo vị trí** (và giữ cặp với tốc độ, không sort riêng)?
- [ ] Duyệt từ xe **gần đích nhất** về xe xa nhất (nếu dùng một biến `pre`)?
- [ ] Điều kiện đoàn mới là `t > pre` (ngặt), gặp đúng tại đích vẫn cùng đoàn?
- [ ] Dùng chia thực `/` (hoặc nhân chéo), không dùng `//`?
- [ ] `pre` khởi tạo `0` (mọi `t > 0` vì `position < target`)?
- [ ] Đã thử: một xe, mọi tốc độ bằng nhau, xe sau nhanh hơn hẳn, `n = 10^5`?
- [ ] Tên method đúng `carFleet(self, target, position, speed)` và trả `int`?